# MD simulation workflow for oligos
This is a quick guide to setting up a linear workflow for running Molecular Dynamics simulations for oligonucleotides.

## Imports
We begin with all top-level requirements and imports, and ensure our software dependencies are setup correctly.

In [2]:
from maize.core.workflow import Workflow
from maize.utilities.io import Config
from maize.steps.mai.gromacs_rna.gmx_rna import MDsRNA
from maize.steps.mai.gromacs.file_utils import SaveFilesFromDict
from maize.steps.mai.gromacs_rna.file_utils import SaveFile
from pathlib import Path
from maize.utilities.execution import JobResourceConfig

## Workflow

Maize will look for configurations of the specific softwares, in particular the locations of the software packages requested. These configuration should be contained in a TOML file. Note that you will need to set up the required software yourself. 

In [5]:
flow = Workflow(name="gmx_rna_MDs", cleanup_temp=False, level="debug")

flow.config = Config()
flow.config.update(Path("gromacs_config.toml"))


Create the node `MDsRNA` that takes pdb structure files as input, prepares topology based on the force-field, and runs MD simulation steps in the following order : Energy minimization -> NVT equilibration -> NPT equilibration -> Production MD. 

In [6]:
md = flow.add(MDsRNA, name="gmx_rna_MDs")

Here we set up paths to input data. `data_dir` should contain input files: pdb structure files to run simulations on. `ff_dir` should contain the force-field folder in the format of GROMACS. List of Gromacs mdp files for MD steps is provided through `mdps`. An `output` folder should be provided wherein the output files get saved.

In [17]:
data_dir = Path("../maize/steps/mai/gromacs_rna/data/")
ff_dir = Path("path/to/ff/mod_amber14sb_OL15.ff")

#Appending rna pdb files to a list
rna_pdbs = []
for num_rna in range(1,4): rna_pdbs.append(Path(f"{data_dir}/rna_pdbs/mol{num_rna}.pdb"))

#Path to mdp file to run 'gmx genion' to include ions in the topology 
mdp_ion = Path("../maize/steps/mai/gromacs_rna/data/mdps/ions.mdp")

#Path to list of mdp files in the following order: energy minimization, NVT equilibration, NPT equilibration, Production MD. The order is important.
mdps = [ Path("..maize/steps/mai/gromacs_rna/data/mdps/em.mdp"), 
          Path("../maize/steps/mai/gromacs_rna/data/mdps/nvt.mdp"),
          Path("../maize/steps/mai/gromacs_rna/data/mdps/npt.mdp"),
          Path("../maize/steps/mai/gromacs_rna/data/mdps/prod.mdp")
          ]

#Path to output folder
output = Path("/path/output") 

### Parameters

Here, we set up parameters for the node. In this example, we set input and parameters to run 2 replicates of MD simulations each for a list of 3 oligo structures. In `sendout_file`, we specify the type of output files we want to save. Note that the names of files in this list should not be changed. The file names following '_', e.g. _tpr, _top, _ndx etc., correspond to the respective file extensions. You can remove some of these file names if you do not want to save those. 

In [18]:
md.inp_rna.set(rna_pdbs)   #List of pdb structure files
md.ff.set(ff_dir)   #Path to Force-field
md.ff_wat.set("tip3p")  #Type of water model
md.box_type.set("cubic")  #Simulation box type
md.replace_with.set("SOL")   #Replacing ions for "SOL"
md.mdp_file.set(mdp_ion)   #MDP to add ions
md.mdp_files.set(mdps)   #MDP file list
md.replicas.set(2)  #2 replicas

#Type of output files to save for each simulation
sendout_file = [
    "topol_tpr",
    "topol_top",
    "confout_gro",
    "ener_edr",
    "md_log",
    "traj_xtc",
    "state_cpt",
    "index_ndx",
    "ch_itp",
    "ch_posre",
]
md.sendout_option.set(sendout_file)

### Nodes to save output files

Creating nodes to save output files. `SaveFile` saves files for each system within the corresponding folder. '.top', '.ndx', '.itp', 'posre.itp' files belong to this type. `SaveFilesFromDict` saves files for each replica per system within the corresponding folder. Files that are specific to replicates : .tpr, .gro, .trr, .xtc, .log, .cpt, .prev.cpt, .edr, belong to this type.

Note that you need to use the same type of saving Node for the files as listed in this example.

In [11]:
save_top = flow.add(SaveFile[dict[str, Path]], name="save_top")
save_ndx = flow.add(SaveFile[dict[str, Path]], name="save_ndx")
save_itp = flow.add(SaveFile[dict[str, list[Path] ]], name="save_itp")
save_posre = flow.add(SaveFile[dict[str, list[Path]]], name="save_posre")
save_tpr = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_tpr")
save_gro = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_gro")
save_trr = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_trr")
save_xtc = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_xtc")
save_log = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_log")
save_cpt = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_cpt")
save_prev_cpt = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_prev_cpt")
save_edr = flow.add(SaveFilesFromDict[dict[tuple[int, str], Path]], name="save_edr")

Sending outputs from the saving nodes to the output destination folder

In [12]:
save_tpr.destination.set(output)
save_top.destination.set(output)
save_itp.destination.set(output)
save_gro.destination.set(output)
save_ndx.destination.set(output)
save_trr.destination.set(output)
save_xtc.destination.set(output)
save_log.destination.set(output)
save_cpt.destination.set(output)
save_prev_cpt.destination.set(output)
save_edr.destination.set(output)
save_posre.destination.set(output)

Output from `MDsRNA` node goes as input to Saving (`SaveFilesFromDict`, `SaveFile`) nodes 

In [13]:
flow.connect(md.out_topol_tpr, save_tpr.inp)
flow.connect(md.out_topol_top, save_top.inp)
flow.connect(md.out_ch_itp, save_itp.inp)
flow.connect(md.out_confout_gro, save_gro.inp)
flow.connect(md.out_index_ndx, save_ndx.inp)
flow.connect(md.out_traj_trr, save_trr.inp)
flow.connect(md.out_traj_xtc, save_xtc.inp)
flow.connect(md.out_md_log, save_log.inp)
flow.connect(md.out_state_cpt, save_cpt.inp)
flow.connect(md.out_state_prev_cpt, save_prev_cpt.inp)
flow.connect(md.out_ener_edr, save_edr.inp)
flow.connect(md.out_ch_posre, save_posre.inp)

### Check
If this method doesn't throw an exception, we have connected everything correctly and set all required parameters.

In [14]:
flow.check() 

### Run
Run the workflow

In [15]:
flow.execute()